# 11 · Custom BaseAgent orchestration

Subclass BaseAgent when Python should decide the execution path. This router reads state and invokes exactly one child.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** 1 Lite call. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


In [ ]:
from google.adk.agents import BaseAgent
from google.adk.events import Event, EventActions

class OrderRouter(BaseAgent):
    async def _run_async_impl(self, ctx):
        order = get_order(ctx.session.state["order_id"])
        route = 0 if order.get("status") == "delivered" else 1
        yield Event(author=self.name, actions=EventActions(state_delta={"routing_reason": order.get("status", "unknown")}))
        async for event in self.sub_agents[route].run_async(ctx):
            yield event

delivered = llm("delivered_help", "Explain that the delivered order may qualify for return review. Do not approve a refund.")
pending = llm("pending_help", "Explain that the order is still processing; do not invent a dispatch date.")
router = OrderRouter(name="deterministic_router", sub_agents=[delivered, pending])
lab = await make_lab(router, state={"order_id": "O1002"})
await ask(lab, "What should I do about this order?")
print(lab.session.state["routing_reason"])


## Try it

Add an explicit unknown-order branch. Compare this routing guarantee with model-selected handoffs.


## Reference

[Official ADK documentation](https://adk.dev/). Shared configuration: `config.json`. No environment activation or login cells are needed.
